Import folder

In [44]:
from pathlib import Path
import pandas as pd
import numpy as np
import pdfplumber
import re

# -----------------------------
# FOLDERS
# -----------------------------

pdf_folder = Path(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\New folder\pdf"
)

pdf_folder = Path(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\New folder\ool"
)

print("PDF files:", len(list(pdf_folder.glob("*.pdf"))))
print("Excel files:", len(list(excel_folder.glob("*.xlsx"))))

PDF files: 55
Excel files: 9


Define months

In [45]:
from pathlib import Path
import pandas as pd
import numpy as np
import pdfplumber
import re

# -----------------------------
# FOLDERS
# -----------------------------

pdf_folder = Path(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\New folder\pdf"
)

excel_folder = Path(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\New folder\excel"
)

print("PDF files:", len(list(pdf_folder.glob("*.pdf"))))
print("Excel files:", len(list(excel_folder.glob("*.xlsx"))))

PDF files: 0
Excel files: 9


In [46]:
month_map = {
    "OCTUBRE": 10,
    "NOVIEMBRE": 11,
    "DICIEMBRE": 12,
    "ENERO": 1,
    "FEBRERO": 2,
    "MARZO": 3,
    "ABRIL": 4,
    "MAYO": 5,
    "JUNIO": 6,
    "JULIO": 7,
    "AGOSTO": 8,
    "SEPTIEMBRE": 9
}

## part A Excel 2020/21 -25/26
We already know these files are much easier and that the relevant sheet is BALANCE EXISTENCIAS FÍSICAS.

Extract the excel files exclude s aceituna de mesa

In [79]:
excel_rows = []

for file in excel_folder.glob("*.xlsx"):

    # -----------------------------------
    # Exclude table-olive reports
    # -----------------------------------
    normalized_name = file.name.lower().replace(" ", "").replace("_", "")

    if (
    "aceitunademesa" in normalized_name
    and "aceitedeoliva" not in normalized_name
):
        print("SKIPPED table olives:", file.name)
        continue

    print("Reading:", file.name)

    xls = pd.ExcelFile(file)

    # -----------------------------------
    # Find balance sheet
    # -----------------------------------
    balance_sheet = None

    for sheet in xls.sheet_names:
        if "BALANCE EXISTENCIAS" in sheet.upper():
            balance_sheet = sheet
            break

    if balance_sheet is None:
        print("   WARNING: balance sheet not found")
        continue

    df = pd.read_excel(
        file,
        sheet_name=balance_sheet,
        header=None
    )

    # -----------------------------------
    # Find rows containing months
    # -----------------------------------
    for _, row in df.iterrows():

        values = row.tolist()

        month_name = None
        month_position = None

        for i, value in enumerate(values):

            if pd.isna(value):
                continue

            value_clean = str(value).strip().upper()

            if value_clean in month_map:
                month_name = value_clean
                month_position = i
                break

        if month_name is None:
            continue

        # Need six cells after month:
        # initial stock
        # production
        # imports
        # domestic market
        # exports
        # final stock

        if month_position + 6 >= len(values):
            continue

        excel_rows.append({
            "month_name": month_name,

            "initial_stock_kt":
                pd.to_numeric(
                    values[month_position + 1],
                    errors="coerce"
                ),

            "production_kt":
                pd.to_numeric(
                    values[month_position + 2],
                    errors="coerce"
                ),

            "imports_kt":
                pd.to_numeric(
                    values[month_position + 3],
                    errors="coerce"
                ),

            "domestic_market_kt":
                pd.to_numeric(
                    values[month_position + 4],
                    errors="coerce"
                ),

            "exports_kt":
                pd.to_numeric(
                    values[month_position + 5],
                    errors="coerce"
                ),

            "final_stock_kt":
                pd.to_numeric(
                    values[month_position + 6],
                    errors="coerce"
                ),

            "source_file": file.name
        })


excel_raw = pd.DataFrame(excel_rows)

print("\nRaw rows:", len(excel_raw))

Reading: INFORME FINAL de la Situación de Mercado del Sector del Aceite de oliva__Campaña 202425_Excel_.xlsx
Reading: INFORME MENSUAL de la Situación de Mercado del Sector del Aceite de Oliva y la Aceituna de Mesa_Julio 2026-Excel.xlsx
SKIPPED table olives: informecierrecampanadelasituaciondemercadodelsectordelaaceitunademesa_agosto2021_excel.xlsx
SKIPPED table olives: informecierrecampanadelasituaciondemercadodelsectordelaaceitunademesa_agosto2022_excel.xlsx
SKIPPED table olives: informecierrecampanadelasituaciondemercadodelsectordelaaceitunademesa_septiembre2023_excel.xlsx
Reading: informecierrecampanadelasituaciondemercadodelsectordelaceitedeoliva_septiembre2021_excel.xlsx
Reading: informecierrecampanadelasituaciondemercadodelsectordelaceitedeoliva_septiembre2022_excel.xlsx
Reading: informecierrecampanadelasituaciondemercadodelsectordelaceitedeoliva_septiembre2023_excel.xlsx
Reading: informecierrecampanadelasituaciondemercadodelsectordelaceitedeoliva_septiembre2024_excel.xlsx

Raw r

Cell 4 — Remove repeated tables

This is important because the same sheet can contain month names more than once.

In [80]:
excel_df = (
    excel_raw
    .drop_duplicates(
        subset=["source_file", "month_name"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Rows after removing repeated months:", len(excel_df))

display(excel_df.head(15))

Rows after removing repeated months: 72


,month_name,initial_stock_kt,production_kt,imports_kt,domestic_market_kt,exports_kt,final_stock_kt,source_file
0,OCTUBRE,186.86893,36.869638,15.916,38.408902,60.278,140.94939,INFORME FINAL de la Situación de Mercado del S...
1,NOVIEMBRE,140.94939,266.670287,23.576,40.438747,59.937,330.81993,INFORME FINAL de la Situación de Mercado del S...
2,DICIEMBRE,330.81993,596.431000,30.592,54.225040,69.723,833.89489,INFORME FINAL de la Situación de Mercado del S...
3,ENERO,833.89489,351.480904,17.385,49.430284,89.304,1064.02651,INFORME FINAL de la Situación de Mercado del S...
4,FEBRERO,1064.02651,154.070397,19.042,40.568917,90.628,1105.94199,INFORME FINAL de la Situación de Mercado del S...
5,MARZO,1105.94199,12.203774,22.052,43.606514,96.981,999.61025,INFORME FINAL de la Situación de Mercado del S...
6,ABRIL,999.61025,3.165106,16.635,35.672966,98.778,884.95939,INFORME FINAL de la Situación de Mercado del S...
7,MAYO,884.95939,0.221644,15.277,36.885764,99.324,764.24827,INFORME FINAL de la Situación de Mercado del S...
8,JUNIO,764.24827,NaN,14.541,37.955560,94.483,646.35071,INFORME FINAL de la Situación de Mercado del S...
9,JULIO,646.35071,NaN,13.110,37.736180,110.997,510.72753,INFORME FINAL de la Situación de Mercado del S...


Cell 5 — Detect campaign year

Your filenames use different conventions, so we'll handle them separately.

In [81]:
def get_campaign_start_year(filename):

    name = (
        filename.lower()
        .replace("_", " ")
    )

    # Example:
    # Campaña 202425
    m = re.search(r"20\d{4}", name)

    if m:
        value = m.group()
        return int(value[:4])

    # September final report:
    # septiembre2024 = campaign 2023/24
    m = re.search(r"septiembre.*?(20\d{2})", name)

    if m:
        return int(m.group(1)) - 1

    # January-August report:
    # julio 2026 = campaign 2025/26
    m = re.search(
        r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto)"
        r".*?(20\d{2})",
        name
    )

    if m:
        return int(m.group(2)) - 1

    # October-December report starts campaign
    m = re.search(
        r"(octubre|noviembre|diciembre).*?(20\d{2})",
        name
    )

    if m:
        return int(m.group(2))

    return np.nan

Apply it

In [82]:
excel_df["campaign_start_year"] = (
    excel_df["source_file"]
    .apply(get_campaign_start_year)
)

display(
    excel_df[
        ["source_file", "campaign_start_year"]
    ].drop_duplicates()
)

,source_file,campaign_start_year
0,INFORME FINAL de la Situación de Mercado del S...,2024
12,INFORME MENSUAL de la Situación de Mercado del...,2025
24,informecierrecampanadelasituaciondemercadodels...,2020
36,informecierrecampanadelasituaciondemercadodels...,2021
48,informecierrecampanadelasituaciondemercadodels...,2022
60,informecierrecampanadelasituaciondemercadodels...,2023


Every olive-oil file should have a campaign year. There should be no NaN.

Cell 6 — Create actual dates

In [83]:
excel_df["month_num"] = (
    excel_df["month_name"]
    .map(month_map)
)

excel_df["year"] = np.where(
    excel_df["month_num"] >= 10,
    excel_df["campaign_start_year"],
    excel_df["campaign_start_year"] + 1
)

excel_df["date"] = pd.to_datetime(
    dict(
        year=excel_df["year"].astype(int),
        month=excel_df["month_num"],
        day=1
    )
)

excel_df["marketing_year"] = (
    excel_df["campaign_start_year"]
    .astype(int)
    .apply(
        lambda y: f"{y}/{str(y + 1)[-2:]}"
    )
)

Cell 7 — Remove months that haven't been reported

For example, August and September 2026 appeared in the July report but contained no observations.

In [84]:
operational_cols = [
    "initial_stock_kt",
    "production_kt",
    "imports_kt",
    "domestic_market_kt",
    "exports_kt",
    "final_stock_kt"
]

excel_df = excel_df.dropna(
    subset=operational_cols,
    how="all"
)

excel_df = (
    excel_df
    .sort_values("date")
    .reset_index(drop=True)
)

display(
    excel_df[
        [
            "date",
            "marketing_year",
            "production_kt",
            "imports_kt",
            "domestic_market_kt",
            "exports_kt",
            "final_stock_kt"
        ]
    ]
)

print("Rows:", len(excel_df))
print("Start:", excel_df["date"].min())
print("End:", excel_df["date"].max())

,date,marketing_year,production_kt,imports_kt,domestic_market_kt,exports_kt,final_stock_kt
0,2020-10-01,2020/21,38.300000,18.400,51.324670,95.700,400.87533
1,2020-11-01,2020/21,244.700000,20.100,50.560500,95.700,519.41483
2,2020-12-01,2020/21,475.300000,20.900,62.651330,83.500,869.46350
3,2021-01-01,2020/21,352.100000,11.700,42.707800,82.500,1108.05570
4,2021-02-01,2020/21,227.400000,16.700,49.396000,104.200,1198.55970
...,...,...,...,...,...,...,...
65,2026-03-01,2025/26,84.993297,27.547,48.161457,97.917,952.05616
66,2026-04-01,2025/26,12.211432,23.062,32.422952,86.625,868.28164
67,2026-05-01,2025/26,1.429309,24.994,30.781539,85.095,778.82841
68,2026-06-01,2025/26,NaN,23.636,34.923590,83.879,683.66182


Rows: 70
Start: 2020-10-01 00:00:00
End: 2026-07-01 00:00:00


At this stage, we should again get approximately what you got before: 70 valid monthly rows through July 2026.

## PART B — PDFs 2016 → 2020

First, let's see whether pdfplumber can extract the actual table structure.

Cell 8 — Inspect ONE PDF table

In [58]:
pdf_folder = Path(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\New folder\ool"
)

In [59]:
print("PDF folder exists:", pdf_folder.exists())

pdf_files = sorted(
    list(pdf_folder.glob("*.pdf")) +
    list(pdf_folder.glob("*.PDF"))
)

print("Number of PDFs:", len(pdf_files))

PDF folder exists: True
Number of PDFs: 110


In [60]:
print("PDF folder:", pdf_folder)
print("Folder exists:", pdf_folder.exists())

print("\nFiles inside folder:")
for f in pdf_folder.iterdir():
    print(f.name)

PDF folder: C:\Users\mt_ha\Desktop\TFM\olive_oil\New folder\ool
Folder exists: True

Files inside folder:
170901datosdeproduccionymovimientodeaceitedeolivayaceitunademesa_datosa31dejuliode2017.pdf
171127_datos_prod_movimientos_existencias_ao_am_octubrede2017.pdf
datos-de-produccion-movimientos_existencias-de-ao_am_agosto-2018.pdf
datosdeproducccionmovimientosyexistenciasdeaceitedeolivayaceitunademesaabril2019.pdf
datosdeproducccionmovimientosyexistenciasdeaceitedeolivayaceitunademesa_marzo2019.pdf
datosdeproduccionexistenciasaoyama31-08-2017.pdf
datosdeproduccionmovimientosyexistenciadeaoyammayo2017.pdf
datosdeproduccionmovimientosyexistenciasdeaceitedeolivayaceitunademesa-julio2018.pdf
datosdeproduccionmovimientosyexistenciasdeaceitedeolivayaceitunademesaabril2017.pdf
datosdeproduccionmovimientosyexistenciasdeaceitedeolivayaceitunademesaabril2021_corregidos.pdf
datosdeproduccionmovimientosyexistenciasdeaceitedeolivayaceitunademesaagosto2019.pdf
datosdeproduccionmovimientosyexistencias

In [61]:
pdf_files = sorted(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

test_pdf = pdf_files[0]

print("Testing:", test_pdf.name)

with pdfplumber.open(test_pdf) as pdf:

    page = pdf.pages[0]

    tables = page.extract_tables()

print("Number of tables detected:", len(tables))

Number of PDFs: 55
Testing: 170901datosdeproduccionymovimientodeaceitedeolivayaceitunademesa_datosa31dejuliode2017.pdf
Number of tables detected: 6


In [62]:
from pathlib import Path

base_folder = Path(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\New folder"
)

print("Folders/files inside New folder:\n")

for f in base_folder.iterdir():
    print(f.name)

Folders/files inside New folder:

6_sep_work.ipynb
excel
ool
Untitled-1.ipynb


In [63]:
for i, table in enumerate(tables):
    print("\n========================")
    print("TABLE", i)
    print("========================")
    
    table_df = pd.DataFrame(table)
    display(table_df)


TABLE 0


,0,1,2,3,4,5,6
0,MES,EXIST. INICIALES,PRODUCCION,IMPORTACIONES,MERCADO INTERIOR\nAPARENTE(1),EXPORTACIONES,EXIST. FINALES
1,OCTUBRE,"331,0","9,7","8,9","35,4","70,1","244,1"
2,NOVIEMBRE,"244,1","93,5","9,7","40,8","78,2","228,3"
3,DICIEMBRE,"228,3","456,3","15,6","47,1","74,2","578,9"
4,ENERO,"578,9","526,2","4,6","53,4","86,7","969,6"
5,FEBRERO,"969,6","155,2","6,3","39,5","96,4","995,2"
6,MARZO,"995,2","37,0","10,0","47,8","99,5","894,9"
7,ABRIL,"894,9","4,5","4,7","31,6","67,0","805,5"
8,MAYO,"805,5","1,2","6,6","34,4","88,9","690,0"
9,JUNIO,"690,0",,"9,4","26,2","80,9","592,3"



TABLE 1


,0,1,2
0,EXPORTACIONES,NaN,NaN
1,PAISES TERCEROS,UE,NaN
2,,ITALIA,OTROS
3,"27,5","17,6","25,0"
4,"24,5","29,6","24,1"
5,"20,7","34,3","19,2"
6,"20,9","43,3","22,5"
7,"21,5","50,6","24,3"
8,"30,2","43,2","26,1"
9,"22,3","23,2","21,5"



TABLE 2


...

,0,1,2
0,IMPORTACIONES,NaN,NaN
1,PAISES TERCEROS,UE,NaN
2,,ITALIA,OTROS
3,"3,3","0,7","4,9"
4,"4,2","1,7","3,8"
5,"6,9","2,4","6,3"
6,"0,3","1,6","2,7"
7,"1,3","1,1","3,9"
8,"3,8","0,9","5,3"
9,"2,0","0,4","2,3"



TABLE 4


,0,1,2,3
0,NaN,VENTAS,OTRAS SALIDAS\nY PERDIDAS,TOTAL
1,NaN,ANIERAC,NaN,
2,OCTUBRE,"25,8","9,6","35,4"
3,NOVIEMBRE,"26,3","14,5","40,8"
4,DICIEMBRE,"23,6","23,5","47,1"
5,ENERO,"22,1","31,3","53,4"
6,FEBRERO,"20,0","19,5","39,5"
7,MARZO,"25,0","22,8","47,8"
8,ABRIL,"22,1","9,5","31,6"
9,MAYO,"22,9","11,5","34,4"



TABLE 5


,0,1,2,3,4,5
0,NaN,ALMAZARAS,PATRIMONIO,ENVASADORAS,TOTAL,
1,NaN,,,OPERADORES,,
2,OCTUBRE,"117,0","4,6","122,5","244,1",
3,NOVIEMBRE,"110,2","2,5","115,6","228,3",
4,DICIEMBRE,"461,7","3,3","113,9","578,9",
5,ENERO,"793,7","17,3","158,6","969,6",
6,FEBRERO,"775,5","32,5","187,2","995,2",
7,MARZO,"665,1","31,5","198,3","894,9",
8,ABRIL,"575,7","28,0","201,8","805,5",
9,MAYO,"468,4","24,0","197,6","690,0",


In [64]:
table0 = pd.DataFrame(tables[0])

print("Shape:", table0.shape)

print("\nColumns / headers:")
for i, value in enumerate(table0.iloc[0]):
    print(i, ":", value)

Shape: (14, 7)

Columns / headers:
0 : MES
1 : EXIST. INICIALES
2 : PRODUCCION
3 : IMPORTACIONES
4 : MERCADO INTERIOR
APARENTE(1)
5 : EXPORTACIONES
6 : EXIST. FINALES


In [65]:
def pdf_num(x):
    if x is None or pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "":
        return np.nan

    x = x.replace(".", "").replace(",", ".")

    try:
        return float(x)
    except:
        return np.nan


pdf_rows = []

for file in pdf_files:

    print("Reading:", file.name)

    try:
        with pdfplumber.open(file) as pdf:

            page = pdf.pages[0]
            tables = page.extract_tables()

            if not tables:
                continue

            table = pd.DataFrame(tables[0])

    except Exception as e:
        print("ERROR:", file.name, e)
        continue

    # Skip if table structure is not the expected 7 columns
    if table.shape[1] != 7:
        continue

    # Loop through monthly rows
    for i in range(1, len(table)):

        month_name = str(table.iloc[i, 0]).strip().upper()

        if month_name not in month_map:
            continue

        pdf_rows.append({
            "month_name": month_name,
            "initial_stock_kt": pdf_num(table.iloc[i, 1]),
            "production_kt": pdf_num(table.iloc[i, 2]),
            "imports_kt": pdf_num(table.iloc[i, 3]),
            "domestic_market_kt": pdf_num(table.iloc[i, 4]),
            "exports_kt": pdf_num(table.iloc[i, 5]),
            "final_stock_kt": pdf_num(table.iloc[i, 6]),
            "source_file": file.name
        })


pdf_raw = pd.DataFrame(pdf_rows)

print("\nRaw PDF rows:", len(pdf_raw))
display(pdf_raw.head(20))

Reading: 170901datosdeproduccionymovimientodeaceitedeolivayaceitunademesa_datosa31dejuliode2017.pdf
Reading: 171127_datos_prod_movimientos_existencias_ao_am_octubrede2017.pdf
Reading: datos-de-produccion-movimientos_existencias-de-ao_am_agosto-2018.pdf
Reading: datos_prod_movimiento_existenciasaceitedeoliva_aceitunademesa_dic2017.pdf
Reading: datos_prod_movimientos_existencias_ao_am_enerode2019.pdf
Reading: datos_prod_movimientos_existencias_ao_amabril2018.pdf
Reading: datos_produccionmovimientos_existenciasdeaoyam_diciembre2018.pdf
Reading: datos_produccionmovimientos_existenciasdeaoyam_marzo2018.pdf
Reading: datos_produccionmovimientosyexistenciasdeao_amfebrero2018.pdf
Reading: datosdeprod_movimientos_existenciasdeao_am_junio2019.pdf
Reading: datosdeproducccionmovimientosyexistenciasdeaceitedeolivayaceitunademesa_marzo2019.pdf
Reading: datosdeproducccionmovimientosyexistenciasdeaceitedeolivayaceitunademesaabril2019.pdf
Reading: datosdeproduccionexistenciasaoyama31-08-2017.pdf
Reading

,month_name,initial_stock_kt,production_kt,imports_kt,domestic_market_kt,exports_kt,final_stock_kt,source_file
0,OCTUBRE,331.0,9.7,8.9,35.4,70.1,244.1,170901datosdeproduccionymovimientodeaceitedeol...
1,NOVIEMBRE,244.1,93.5,9.7,40.8,78.2,228.3,170901datosdeproduccionymovimientodeaceitedeol...
2,DICIEMBRE,228.3,456.3,15.6,47.1,74.2,578.9,170901datosdeproduccionymovimientodeaceitedeol...
3,ENERO,578.9,526.2,4.6,53.4,86.7,969.6,170901datosdeproduccionymovimientodeaceitedeol...
4,FEBRERO,969.6,155.2,6.3,39.5,96.4,995.2,170901datosdeproduccionymovimientodeaceitedeol...
5,MARZO,995.2,37.0,10.0,47.8,99.5,894.9,170901datosdeproduccionymovimientodeaceitedeol...
6,ABRIL,894.9,4.5,4.7,31.6,67.0,805.5,170901datosdeproduccionymovimientodeaceitedeol...
7,MAYO,805.5,1.2,6.6,34.4,88.9,690.0,170901datosdeproduccionymovimientodeaceitedeol...
8,JUNIO,690.0,NaN,9.4,26.2,80.9,592.3,170901datosdeproduccionymovimientodeaceitedeol...
9,JULIO,592.3,NaN,7.0,107.2,NaN,492.1,170901datosdeproduccionymovimientodeaceitedeol...


In [68]:
operational_cols = [
    "initial_stock_kt",
    "production_kt",
    "imports_kt",
    "domestic_market_kt",
    "exports_kt",
    "final_stock_kt"
]

pdf_check = (
    pdf_raw
    .assign(
        has_data=lambda x:
            x[operational_cols].notna().any(axis=1)
    )
    .groupby("source_file")
    .agg(
        months_with_data=("has_data", "sum"),
        total_month_rows=("month_name", "size")
    )
    .reset_index()
)

display(pdf_check)

,source_file,months_with_data,total_month_rows
0,170901datosdeproduccionymovimientodeaceitedeol...,10,12
1,171127_datos_prod_movimientos_existencias_ao_a...,1,12
2,datos-de-produccion-movimientos_existencias-de...,11,12
3,datos_prod_movimiento_existenciasaceitedeoliva...,3,12
4,datos_prod_movimientos_existencias_ao_am_enero...,4,12
5,datos_prod_movimientos_existencias_ao_amabril2...,7,12
6,datos_produccionmovimientos_existenciasdeaoyam...,3,12
7,datos_produccionmovimientos_existenciasdeaoyam...,6,12
8,datos_produccionmovimientosyexistenciasdeao_am...,5,12
9,datosdeprod_movimientos_existenciasdeao_am_jun...,9,12


In [69]:
campaign_rows = []

for file in pdf_files:

    try:
        with pdfplumber.open(file) as pdf:
            text = pdf.pages[0].extract_text() or ""

        match = re.search(
            r"CAMPA[ÑN]A\s+(\d{4})\s*/\s*(\d{2,4})",
            text,
            flags=re.IGNORECASE
        )

        if match:
            start_year = int(match.group(1))
            marketing_year = f"{start_year}/{str(start_year + 1)[-2:]}"
        else:
            start_year = np.nan
            marketing_year = None

        campaign_rows.append({
            "source_file": file.name,
            "campaign_start_year": start_year,
            "marketing_year": marketing_year
        })

    except Exception as e:
        print("ERROR:", file.name, e)


campaign_info = pd.DataFrame(campaign_rows)

# Combine with the check we already created
campaign_check = pdf_check.merge(
    campaign_info,
    on="source_file",
    how="left"
)

display(
    campaign_check[
        [
            "marketing_year",
            "months_with_data",
            "source_file"
        ]
    ].sort_values(
        ["marketing_year", "months_with_data"]
    )
)

,marketing_year,months_with_data,source_file
44,2016/17,1,datosdeproduccionmovimientosyexistenciasdeacei...
39,2016/17,2,datosdeproduccionmovimientosyexistenciasdeacei...
22,2016/17,3,datosdeproduccionmovimientosyexistenciasdeacei...
25,2016/17,4,datosdeproduccionmovimientosyexistenciasdeacei...
27,2016/17,5,datosdeproduccionmovimientosyexistenciasdeacei...
50,2016/17,6,datosdeproduccionmovimientosyexistenciasdeaoya...
18,2016/17,7,datosdeproduccionmovimientosyexistenciasdeacei...
13,2016/17,8,datosdeproduccionmovimientosyexistenciadeaoyam...
54,2016/17,9,datosproduccionmovimientosaoyamjunio2017.pdf
0,2016/17,10,170901datosdeproduccionymovimientodeaceitedeol...


In [70]:
selected_reports = (
    campaign_check
    .sort_values(
        ["marketing_year", "months_with_data"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset="marketing_year",
        keep="first"
    )
    .reset_index(drop=True)
)

display(
    selected_reports[
        [
            "marketing_year",
            "campaign_start_year",
            "months_with_data",
            "source_file"
        ]
    ]
)

,marketing_year,campaign_start_year,months_with_data,source_file
0,2016/17,2016,12,datosdeproduccionymovimientosdeaceitedeolivaya...
1,2017/18,2017,12,datosdeproduccionmovimientosyexistenciasdeacei...
2,2018/19,2018,12,datosdeproduccionmovimientosyexistenciasdeacei...
3,2019/20,2019,12,datosdeproduccionmovimientosyexistenciasdeacei...
4,2020/21,2020,8,datosdeproduccionmovimientosyexistenciasdeacei...


In [71]:
# Keep only complete PDF campaigns before the Excel period
selected_complete = selected_reports[
    selected_reports["marketing_year"].isin(
        ["2016/17", "2017/18", "2018/19", "2019/20"]
    )
].copy()


# Get selected filenames
selected_files = selected_complete["source_file"].tolist()


# Keep rows only from those selected reports
pdf_df = pdf_raw[
    pdf_raw["source_file"].isin(selected_files)
].copy()


# Add campaign information
pdf_df = pdf_df.merge(
    selected_complete[
        [
            "source_file",
            "campaign_start_year",
            "marketing_year"
        ]
    ],
    on="source_file",
    how="left"
)


# Create month number
pdf_df["month_num"] = pdf_df["month_name"].map(month_map)


# Calendar year:
# Oct-Dec = campaign start year
# Jan-Sep = next year
pdf_df["year"] = np.where(
    pdf_df["month_num"] >= 10,
    pdf_df["campaign_start_year"],
    pdf_df["campaign_start_year"] + 1
)


# Create date
pdf_df["date"] = pd.to_datetime(
    dict(
        year=pdf_df["year"].astype(int),
        month=pdf_df["month_num"],
        day=1
    )
)


# Sort
pdf_df = (
    pdf_df
    .sort_values("date")
    .reset_index(drop=True)
)


print("PDF rows:", len(pdf_df))
print("Start:", pdf_df["date"].min())
print("End:", pdf_df["date"].max())

display(
    pdf_df[
        [
            "date",
            "marketing_year",
            "initial_stock_kt",
            "production_kt",
            "imports_kt",
            "domestic_market_kt",
            "exports_kt",
            "final_stock_kt"
        ]
    ]
)

PDF rows: 48
Start: 2016-10-01 00:00:00
End: 2020-09-01 00:00:00


,date,marketing_year,initial_stock_kt,production_kt,imports_kt,domestic_market_kt,exports_kt,final_stock_kt
0,2016-10-01,2016/17,331.0,9.7,8.9,35.4,70.1,244.1
1,2016-11-01,2016/17,244.1,93.5,9.7,40.8,78.2,228.3
2,2016-12-01,2016/17,228.3,456.2,15.6,47.0,74.2,578.9
3,2017-01-01,2016/17,578.9,526.2,4.6,53.5,86.7,969.5
4,2017-02-01,2016/17,969.5,155.3,6.3,39.7,96.4,995.0
5,2017-03-01,2016/17,995.0,37.0,10.0,47.8,99.5,894.7
6,2017-04-01,2016/17,894.7,4.5,4.7,31.5,67.0,805.4
7,2017-05-01,2016/17,805.4,4.2,6.6,37.4,88.9,689.9
8,2017-06-01,2016/17,689.9,NaN,9.4,26.5,80.9,591.9
9,2017-07-01,2016/17,591.9,NaN,5.3,29.8,75.6,491.8


In [72]:
# Check missing months in PDF period

expected_pdf = pd.date_range(
    start=pdf_df["date"].min(),
    end=pdf_df["date"].max(),
    freq="MS"
)

missing_pdf = expected_pdf.difference(pdf_df["date"])

print("Expected months:", len(expected_pdf))
print("Actual rows:", len(pdf_df))
print("Missing months:", list(missing_pdf))

print("\nDuplicate dates:")
display(
    pdf_df[pdf_df.duplicated("date", keep=False)]
)

Expected months: 48
Actual rows: 48
Missing months: []

Duplicate dates:


,month_name,initial_stock_kt,production_kt,imports_kt,domestic_market_kt,exports_kt,final_stock_kt,source_file,campaign_start_year,marketing_year,month_num,year,date


In [73]:
check_cols = [
    "initial_stock_kt",
    "production_kt",
    "imports_kt",
    "domestic_market_kt",
    "exports_kt",
    "final_stock_kt"
]

print(pdf_df[check_cols].isna().sum())

initial_stock_kt       0
production_kt         16
imports_kt             0
domestic_market_kt     0
exports_kt             4
final_stock_kt         0
dtype: int64


In [74]:
display(
    pdf_df[
        pdf_df[
            [
                "production_kt",
                "exports_kt"
            ]
        ].isna().any(axis=1)
    ][
        [
            "date",
            "marketing_year",
            "production_kt",
            "exports_kt",
            "initial_stock_kt",
            "imports_kt",
            "domestic_market_kt",
            "final_stock_kt"
        ]
    ]
)

,date,marketing_year,production_kt,exports_kt,initial_stock_kt,imports_kt,domestic_market_kt,final_stock_kt
8,2017-06-01,2016/17,NaN,80.9,689.9,9.4,26.5,591.9
9,2017-07-01,2016/17,NaN,75.6,591.9,5.3,29.8,491.8
10,2017-08-01,2016/17,NaN,61.8,491.8,6.2,36.6,399.6
11,2017-09-01,2016/17,NaN,NaN,399.6,6.0,101.2,304.4
20,2018-06-01,2017/18,NaN,93.5,811.7,13.3,30.6,700.9
21,2018-07-01,2017/18,NaN,88.0,700.9,11.9,42.5,582.3
22,2018-08-01,2017/18,NaN,71.3,582.3,14.5,39.7,485.8
23,2018-09-01,2017/18,NaN,NaN,485.8,11.0,122.0,374.8
32,2019-06-01,2018/19,NaN,90.3,1242.3,9.9,44.6,1117.3
33,2019-07-01,2018/19,NaN,103.4,1117.3,8.8,39.7,983.0


In [85]:
display(
    excel_df[
        excel_df["month_num"].isin([6, 7, 8, 9])
    ][
        [
            "date",
            "marketing_year",
            "production_kt",
            "exports_kt"
        ]
    ].sort_values("date")
)

,date,marketing_year,production_kt,exports_kt
8,2021-06-01,2020/21,NaN,87.800000
9,2021-07-01,2020/21,NaN,81.900000
10,2021-08-01,2020/21,NaN,66.000000
11,2021-09-01,2020/21,NaN,103.800000
20,2022-06-01,2021/22,NaN,103.446320
21,2022-07-01,2021/22,NaN,90.426970
22,2022-08-01,2021/22,NaN,83.145230
23,2022-09-01,2021/22,NaN,89.365840
32,2023-06-01,2022/23,NaN,65.064920
33,2023-07-01,2022/23,NaN,58.873541


In [86]:
cols = [
    "date",
    "marketing_year",
    "initial_stock_kt",
    "production_kt",
    "imports_kt",
    "domestic_market_kt",
    "exports_kt",
    "final_stock_kt"
]

pdf_clean = pdf_df[cols].copy()
excel_clean = excel_df[cols].copy()

operational_df = pd.concat(
    [pdf_clean, excel_clean],
    ignore_index=True
)

operational_df = (
    operational_df
    .sort_values("date")
    .drop_duplicates(subset="date", keep="last")
    .reset_index(drop=True)
)

print("Rows:", len(operational_df))
print("Start:", operational_df["date"].min())
print("End:", operational_df["date"].max())

print("\nMissing values:")
print(operational_df[cols[2:]].isna().sum())

Rows: 118
Start: 2016-10-01 00:00:00
End: 2026-07-01 00:00:00

Missing values:
initial_stock_kt       0
production_kt         38
imports_kt             0
domestic_market_kt     0
exports_kt             4
final_stock_kt         0
dtype: int64


In [87]:
print("Excel start:", excel_df["date"].min())
print("Excel end:", excel_df["date"].max())

print("\nFiles currently represented in excel_df:")

display(
    excel_df[
        ["source_file", "marketing_year"]
    ]
    .drop_duplicates()
)

Excel start: 2020-10-01 00:00:00
Excel end: 2026-07-01 00:00:00

Files currently represented in excel_df:


,source_file,marketing_year
0,informecierrecampanadelasituaciondemercadodels...,2020/21
12,informecierrecampanadelasituaciondemercadodels...,2021/22
24,informecierrecampanadelasituaciondemercadodels...,2022/23
36,informecierrecampanadelasituaciondemercadodels...,2023/24
48,INFORME FINAL de la Situación de Mercado del S...,2024/25
60,INFORME MENSUAL de la Situación de Mercado del...,2025/26


In [88]:
expected_all = pd.date_range(
    start=operational_df["date"].min(),
    end=operational_df["date"].max(),
    freq="MS"
)

missing_all = expected_all.difference(operational_df["date"])

print("Rows:", len(operational_df))
print("Start:", operational_df["date"].min())
print("End:", operational_df["date"].max())
print("Missing months:", list(missing_all))
print("Duplicate dates:", operational_df["date"].duplicated().sum())

Rows: 118
Start: 2016-10-01 00:00:00
End: 2026-07-01 00:00:00
Missing months: []
Duplicate dates: 0


In [89]:
operational_raw = operational_df.copy()

operational_model = operational_df.copy()

In [90]:
# Keep the raw version untouched
operational_raw = operational_df.copy()

# Export CSV
operational_raw.to_csv(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\operational_monthly_2016_2026.csv",
    index=False
)

# Export Excel too
operational_raw.to_excel(
    r"C:\Users\mt_ha\Desktop\TFM\olive_oil\operational_monthly_2016_2026.xlsx",
    index=False
)

print("Saved!")
print(operational_raw.shape)

Saved!
(118, 8)
